# Section 1: Setup & Data Preparation
This section loads CIFAR-10 (you can swap in the Cats vs. Dogs dataset using torchvision.datasets.ImageFolder). To ensure you can overfit and observe double descent, data augmentation is kept minimal.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Set a fixed seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# If you are using a GPU, you must also seed CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Forces deterministic algorithms (can slightly slow down training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Basic transforms (No heavy augmentation to allow intentional overfitting)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

100%|██████████| 170M/170M [06:00<00:00, 473kB/s]


# Section 2: Shared Trunk & Tree Node Architecture
To satisfy the requirement of providing appropriate features, this uses a shared convolutional trunk. The parent node makes an initial prediction. Child nodes can act as "expert" classifiers to correct specific misclassifications, mirroring the logic of adding child neurons to fix specific error categories.

In [ ]:
class FeatureTrunk(nn.Module):
    """Shared feature extractor for all nodes in the tree."""
    def __init__(self):
        super(OptimizedFeatureTrunk, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 4 (Crucial for larger Cats vs Dogs images)
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)) # Forces output to a fixed 4x4 spatial size
        )

    def forward(self, x):
        x = self.features(x)
        return torch.flatten(x, 1) # Output dimension will be 128 * 4 * 4 = 2048

class TreeNode(nn.Module):
    """A small neural network acting as a single node in the tree."""
    def __init__(self, input_dim, output_classes):
        super(TreeNode, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            # No Dropout - we want to overfit
            nn.Linear(128, output_classes)
        )

    def forward(self, x):
        return self.classifier(x)

class TreeNetwork(nn.Module):
    def __init__(self, num_classes=10, num_child_nodes=2, neurons_per_node=128):
        super(ScalableTreeNetwork, self).__init__()

        # 1. The Shared Trunk (using the optimized version)
        self.trunk = OptimizedFeatureTrunk()
        trunk_out_dim = 128 * 4 * 4 # 2048

        # 2. The Parent Node
        # Increasing 'neurons_per_node' adds non-linear basis functions to drive overfitting
        self.parent_node = nn.Sequential(
            nn.Linear(trunk_out_dim, neurons_per_node),
            nn.ReLU(),
            nn.Linear(neurons_per_node, num_classes)
        )

        # 3. Scalable Child Nodes (ModuleList allows dynamic scaling)
        # To push past the interpolation regime, increase 'num_child_nodes'
        self.children = nn.ModuleList([
            nn.Sequential(
                nn.Linear(trunk_out_dim, neurons_per_node),
                nn.ReLU(),
                nn.Linear(neurons_per_node, num_classes)
            ) for _ in range(num_child_nodes)
        ])

    def forward(self, x):
        features = self.trunk(x)
        parent_out = self.parent_node(features)

        # BASE ROUTING LOGIC:
        # In a fully realized tree, you would route specific features to specific
        # children based on C3/C4 error categories during training.
        # For a basic ensemble sum to brute-force complexity:
        child_outputs = torch.zeros_like(parent_out)
        for child in self.children:
            child_outputs += child(features)

        return parent_out + child_outputs

# To observe double descent, loop through increasing complexities:
# e.g., models with 2, 5, 10, and 20 child nodes, tracking test accuracy for each.
# net = ScalableTreeNetwork(num_child_nodes=10, neurons_per_node=256).to(device)

net = TreeNetwork().to(device)

# Section 3: Training Loop (Interpolation / Overfitting)
To overfit and reach near 100% training accuracy, use a high learning rate initially, no weight decay (L2 regularization), and train for an extended number of epochs to push past the interpolation threshold into the double descent regime.

In [ ]:
criterion = nn.CrossEntropyLoss()
# No weight_decay (regularization) to encourage overfitting
optimizer = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)

epochs = 150 # High number of epochs to observe double descent

train_acc_history = []
test_acc_history = []

for epoch in range(epochs):
    net.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_acc_history.append(train_acc)

    # Evaluate on test set to track double descent
    net.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = 100 * test_correct / test_total
    test_acc_history.append(test_acc)

    print(f'Epoch {epoch+1} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%')

Epoch 1 | Train Acc: 47.63% | Test Acc: 59.31%
Epoch 2 | Train Acc: 64.79% | Test Acc: 66.65%
Epoch 3 | Train Acc: 71.62% | Test Acc: 68.30%
Epoch 4 | Train Acc: 76.93% | Test Acc: 68.95%
Epoch 5 | Train Acc: 81.57% | Test Acc: 69.71%
Epoch 6 | Train Acc: 85.80% | Test Acc: 69.70%
Epoch 7 | Train Acc: 88.70% | Test Acc: 69.18%
Epoch 8 | Train Acc: 90.82% | Test Acc: 68.35%
Epoch 9 | Train Acc: 92.10% | Test Acc: 68.27%
Epoch 10 | Train Acc: 93.18% | Test Acc: 69.38%
Epoch 11 | Train Acc: 94.31% | Test Acc: 68.96%
Epoch 12 | Train Acc: 93.98% | Test Acc: 69.29%
Epoch 13 | Train Acc: 94.46% | Test Acc: 68.22%
Epoch 14 | Train Acc: 94.79% | Test Acc: 68.93%
Epoch 15 | Train Acc: 94.73% | Test Acc: 68.31%
Epoch 16 | Train Acc: 94.92% | Test Acc: 68.56%
Epoch 17 | Train Acc: 95.31% | Test Acc: 68.01%
Epoch 18 | Train Acc: 95.66% | Test Acc: 67.64%
Epoch 19 | Train Acc: 95.61% | Test Acc: 68.36%
Epoch 20 | Train Acc: 96.14% | Test Acc: 66.84%
Epoch 21 | Train Acc: 95.58% | Test Acc: 67.27%
E

KeyboardInterrupt: 

# Section 4: Receptive Field Visualization (Saliency Maps)
To visualize the receptive fields of the nodes, tracking the gradient of the target class back to the input image (Saliency Mapping) highlights exactly which pixels the network node is "looking at."

In [ ]:
def visualize_receptive_field(model, image_tensor, target_class):
    """Visualizes the receptive field via input gradients (Saliency Map)."""
    model.eval()
    image_tensor = image_tensor.to(device).requires_grad_()

    # Forward pass
    output = model(image_tensor)

    # Target the specific class
    score = output[0, target_class]
    model.zero_grad()
    score.backward()

    # Get the gradients of the input image
    saliency, _ = torch.max(image_tensor.grad.data.abs(), dim=1)
    saliency = saliency.squeeze().cpu().numpy()

    # Plot
    plt.imshow(saliency, cmap='hot')
    plt.title(f'Receptive Field Saliency (Class {target_class})')
    plt.axis('off')
    plt.show()

# Usage: Pass a single batch image through the function
# sample_img = next(iter(trainloader))[0][0].unsqueeze(0)
# visualize_receptive_field(net, sample_img, target_class=3)

# Section 5: Minimizing Network Size (Pruning)
To minimize the size of the network, you can apply L1 Unstructured Pruning to the dense layers in your nodes, removing the lowest-magnitude weights without altering the architecture shape.

In [ ]:
import torch.nn.utils.prune as prune

def prune_tree_network(model, pruning_amount=0.3):
    """Applies L1 unstructured pruning to the linear layers of the nodes."""
    for name, module in model.named_modules():
        # Target the dense classification layers in the tree nodes
        if isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=pruning_amount)
            # Make pruning permanent
            prune.remove(module, 'weight')
    print(f"Network pruned by {pruning_amount*100}%")

# Usage: Call this after training is complete, then fine-tune if necessary.
# prune_tree_network(net, pruning_amount=0.4)